# Notebook 2: Baseline mBERT Parser

**Goal**: Reproduce the baseline results from Table 2 of Chau et al. (2020).

We train the biaffine dependency parser with mBERT in two variants:
- **MBERT (frozen)**: mBERT weights are fixed; a BiLSTM processes the output
- **MBERT + FT**: mBERT weights are fine-tuned during parser training; no BiLSTM

**Expected results from the paper:**

| Language | MBERT (frozen) | MBERT + FT |
|---|---|---|
| Irish (GA) | 68.19 | 72.67 |
| Maltese (MT) | 67.06 | 76.74 |
| Singlish (SING) | 74.01 | 78.24 |
| Vietnamese (VI) | 62.96 | 66.13 |

**Prerequisites**: Run Notebook 01 first!

In [ ]:
# Environment setup (run this every time you open Colab)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR  = '/content/parsing-mbert'
WORKSPACE = '/content/drive/MyDrive/thesis_mbert'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/ethch18/parsing-mbert.git {REPO_DIR}')

sys.path.insert(0, os.path.join(REPO_DIR, 'thesis', 'src'))

!pip install -q transformers==4.38.0 tokenizers datasets peft accelerate conllu tqdm sentencepiece

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Data paths
UD_BASE = os.path.join(WORKSPACE, 'ud-treebanks-v2.5')

TREEBANKS = {
    'ga':   {'train': f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-test.conllu'},
    'mt':   {'train': f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-test.conllu'},
    'vi':   {'train': f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-test.conllu'},
    'sing': {'train': f'{REPO_DIR}/data/sing/train.conllu',
             'dev':   f'{REPO_DIR}/data/sing/dev.conllu',
             'test':  f'{REPO_DIR}/data/sing/test.conllu'},
}
print('Paths configured.')

In [ ]:
# Core training function
# This runs one full experiment: load data, build model, train, evaluate

import json
from transformers import AutoTokenizer
from data.conllu_reader import read_conllu, get_relation_vocab
from models.encoder import BERTEncoder
from models.biaffine_parser import BiaffineParser
from training.trainer import ParserTrainer

def run_experiment(
    lang: str,
    model_name: str = 'bert-base-multilingual-cased',
    freeze_bert: bool = True,
    experiment_name: str = 'mbert_frozen',
    max_epochs: int = 200,
    patience: int = 20,
    batch_size: int = 24,
    bert_lr: float = 5e-5,
    parser_lr: float = 1e-3,
):
    """
    Run a full parse training experiment.

    Args:
        lang: language code ('ga', 'mt', 'vi', 'sing')
        model_name: HuggingFace model ID or local path
        freeze_bert: if True, freeze BERT weights (non-FT variant)
        experiment_name: name used for saving results
        ...

    Returns:
        dict with test LAS and UAS
    """
    print(f'\n{"="*60}')
    print(f'Experiment: {experiment_name} | Language: {lang.upper()}')
    print(f'Model: {model_name}  |  Freeze BERT: {freeze_bert}')
    print(f'{"="*60}\n')

    paths = TREEBANKS[lang]

    # Load data
    train_sents = read_conllu(paths['train'])
    dev_sents   = read_conllu(paths['dev'])
    test_sents  = read_conllu(paths['test'])

    print(f'Train: {len(train_sents)} sentences')
    print(f'Dev:   {len(dev_sents)} sentences')
    print(f'Test:  {len(test_sents)} sentences')

    # Build relation vocab from training data
    rel_vocab = get_relation_vocab(train_sents)
    n_rels = max(rel_vocab.values()) + 1
    print(f'Relations: {n_rels}')

    # Load tokenizer and encoder
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    encoder = BERTEncoder(model_name=model_name, freeze=freeze_bert, dropout=0.1)

    # For frozen variant: use BiLSTM; for FT variant: no BiLSTM (direct biaffine)
    use_bilstm = freeze_bert

    model = BiaffineParser(
        encoder=encoder,
        n_rels=n_rels,
        arc_dim=500,
        rel_dim=100,
        bilstm_layers=3,
        bilstm_hidden=400,
        mlp_dropout=0.33,
        use_bilstm=use_bilstm,
    )

    total_params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total params: {total_params:,}  |  Trainable: {trainable:,}')

    # Save directory
    save_dir = os.path.join(WORKSPACE, 'results', experiment_name, lang)
    os.makedirs(save_dir, exist_ok=True)

    # Train
    trainer = ParserTrainer(
        model=model,
        train_sentences=train_sents,
        dev_sentences=dev_sents,
        tokenizer=tokenizer,
        rel_vocab=rel_vocab,
        save_dir=save_dir,
        batch_size=batch_size,
        max_epochs=max_epochs,
        patience=patience,
        bert_lr=bert_lr,
        parser_lr=parser_lr,
        device=DEVICE,
    )

    best_dev_las = trainer.train()

    # Evaluate on test set
    test_las, test_uas = trainer.evaluate_test(test_sents, tokenizer)

    result = {
        'lang': lang,
        'experiment': experiment_name,
        'model': model_name,
        'freeze_bert': freeze_bert,
        'best_dev_las': round(best_dev_las, 2),
        'test_las': round(test_las, 2),
        'test_uas': round(test_uas, 2),
    }

    # Save result
    result_path = os.path.join(save_dir, 'result.json')
    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)

    print(f'\nFinal test result saved to: {result_path}')
    return result

In [ ]:
# Run MBERT Frozen on all 4 languages
# Each language takes roughly 30-60 minutes on a T4 GPU

LANGS = ['ga', 'mt', 'vi', 'sing']
frozen_results = {}

for lang in LANGS:
    result = run_experiment(
        lang=lang,
        model_name='bert-base-multilingual-cased',
        freeze_bert=True,
        experiment_name='mbert_frozen',
        max_epochs=200,
        patience=20,
    )
    frozen_results[lang] = result
    print(f'\n[{lang.upper()}] Test LAS = {result["test_las"]}')

In [ ]:
# Run MBERT + FT (fine-tuned) on all 4 languages
# FT variant: no BiLSTM, BERT weights are updated
# Typically takes 30-90 minutes per language

ft_results = {}

for lang in LANGS:
    result = run_experiment(
        lang=lang,
        model_name='bert-base-multilingual-cased',
        freeze_bert=False,
        experiment_name='mbert_ft',
        max_epochs=200,
        patience=20,
        bert_lr=5e-5,
        parser_lr=1e-3,
    )
    ft_results[lang] = result
    print(f'\n[{lang.upper()}] Test LAS = {result["test_las"]}')

In [ ]:
# Print summary table comparing our results vs the paper

PAPER_RESULTS = {
    'mbert_frozen': {'ga': 68.19, 'mt': 67.06, 'sing': 74.01, 'vi': 62.96},
    'mbert_ft':     {'ga': 72.67, 'mt': 76.74, 'sing': 78.24, 'vi': 66.13},
}

LANG_NAMES = {'ga': 'Irish', 'mt': 'Maltese', 'vi': 'Vietnamese', 'sing': 'Singlish'}

print(f'{"-"*70}')
print(f'{"Experiment":<20} {"Language":<12} {"Ours (LAS)":>12} {"Paper (LAS)":>12} {"Diff":>8}')
print(f'{"-"*70}')

for exp_name, results in [('MBERT frozen', frozen_results), ('MBERT + FT', ft_results)]:
    key = 'mbert_frozen' if 'frozen' in exp_name else 'mbert_ft'
    for lang in LANGS:
        if lang in results:
            ours = results[lang]['test_las']
            paper = PAPER_RESULTS[key].get(lang, 0)
            diff = ours - paper
            print(f'{exp_name:<20} {LANG_NAMES[lang]:<12} {ours:>12.2f} {paper:>12.2f} {diff:>+8.2f}')

print(f'{"-"*70}')
print('\nNote: Small differences from the paper are expected due to randomness.')
print('The paper averages 5 random seeds; we run 1 seed by default.')